# 02 — Enrich dim_listing (build tier classification)

**Input:** `cache_drive/dim_listing.parquet`, `events_positive.parquet`, `snapshot_60d.parquet`.
**Output:** `cache_drive/dim_listing_enriched.parquet` với:
- aggregates: `first_evt_date`, `last_evt_date`, `n_pos_train`, `n_unique_users`
- snapshot agg: `last_snap_date`, `n_snap_days`, `views_total`, `contacts_total`
- flags: `is_pre_fact_window`, `has_any_event_train`, `is_dead`
- `recency_evt_days`, `age_at_train_end`, `tier ∈ {A, B, C, D}`

Đọc cache, ghi cache. Không đụng tới `Datathon_Data/`.

In [ ]:
# Local kernel: assume deps already installed.
# To install run once:
#   pip install pyarrow pandas numpy scipy lightgbm scikit-learn tqdm
print("Skipping pip install (local kernel).")

In [ ]:
# === Local setup: paths + add training/utils to sys.path ===
import os, sys

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
DATA_DIR     = os.path.join(PROJECT_DIR, "Datathon_Data")  # contains train/ and test/
os.makedirs(CACHE_DIR, exist_ok=True)

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("CACHE_DIR  :", CACHE_DIR)
print("utils available:", os.path.isdir(os.path.join(TRAINING_DIR, "utils")))

In [ ]:
TRAIN_DATE_START = "2025-11-09"
TRAIN_DATE_END = "2026-04-09"
VALID_START = "2026-03-13"

POSITIVE_EVENT_TYPES = frozenset({
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms",
})
HIGH_INTENT_EVENTS = frozenset({"view_phone", "contact_chat", "contact_zalo", "contact_sms"})

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}

# Local data paths (relative to DATA_DIR defined in the setup cell)
TRAIN_PATH = os.path.join(DATA_DIR, "train") + os.sep
TEST_PATH  = os.path.join(DATA_DIR, "test", "test") + os.sep

print("Constants loaded. TRAIN_END:", TRAIN_DATE_END, "VALID_START:", VALID_START)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

In [ ]:
# Local mode: no GCS egress guardrail needed.
print("Local kernel: reading from CACHE_DIR.")

In [ ]:
import time
import numpy as np
import pandas as pd

t0 = time.time()
dim = pd.read_parquet(f"{CACHE_DIR}/dim_listing.parquet")
print(f"dim: {dim.shape} | {time.time()-t0:.1f}s")
dim["posted_date"] = pd.to_datetime(dim["posted_date"], errors="coerce")

In [ ]:
# --- aggregate events_positive per item (pyarrow native group_by) ---
import pyarrow as pa, pyarrow.dataset as ds
t0 = time.time()
_evt_dset = ds.dataset(f"{CACHE_DIR}/events_positive.parquet", format="parquet")
_evt_tbl = _evt_dset.to_table(columns=["item_id", "user_id", "date"])
_g = _evt_tbl.group_by("item_id").aggregate([
    ("user_id", "count"),
    ("user_id", "count_distinct"),
    ("date",    "min"),
    ("date",    "max"),
])
evt_agg = _g.rename_columns([
    "item_id", "n_pos_train", "n_unique_users", "first_evt_date", "last_evt_date",
]).to_pandas()
evt_agg["first_evt_date"] = pd.to_datetime(evt_agg["first_evt_date"], errors="coerce")
evt_agg["last_evt_date"]  = pd.to_datetime(evt_agg["last_evt_date"],  errors="coerce")
print(f"evt_agg: {evt_agg.shape} | {time.time()-t0:.1f}s")
del _evt_tbl, _g

In [ ]:
# --- aggregate snapshot (pyarrow native group_by) ---
t0 = time.time()
_snap_dset = ds.dataset(f"{CACHE_DIR}/snapshot_60d.parquet", format="parquet")
_snap_tbl = _snap_dset.to_table(columns=["item_id", "date", "views_24h", "contacts_24h"])
_g = _snap_tbl.group_by("item_id").aggregate([
    ("date",         "max"),
    ("date",         "count_distinct"),
    ("views_24h",    "sum"),
    ("contacts_24h", "sum"),
])
snap_agg = _g.rename_columns([
    "item_id", "last_snap_date", "n_snap_days", "views_total", "contacts_total",
]).to_pandas()
snap_agg["last_snap_date"] = pd.to_datetime(snap_agg["last_snap_date"], errors="coerce")
print(f"snap_agg: {snap_agg.shape} | {time.time()-t0:.1f}s")
del _snap_tbl, _g

In [ ]:
# --- merge + flags + tier ---
TRAIN_END_TS = pd.Timestamp(TRAIN_DATE_END)
FACT_START_TS = pd.Timestamp("2025-11-09")

enr = dim.merge(evt_agg, on="item_id", how="left")
enr = enr.merge(snap_agg, on="item_id", how="left")

for c in ("n_pos_train", "n_unique_users", "n_snap_days",
          "views_total", "contacts_total"):
    enr[c] = enr[c].fillna(0).astype("int64")

enr["age_at_train_end"] = (TRAIN_END_TS - enr["posted_date"]).dt.days
enr["is_pre_fact_window"] = (enr["posted_date"] < FACT_START_TS).astype("int8")
enr["has_any_event_train"] = (enr["n_pos_train"] > 0).astype("int8")
enr["last_evt_date"] = pd.to_datetime(enr["last_evt_date"], errors="coerce")
enr["recency_evt_days"] = (TRAIN_END_TS - enr["last_evt_date"]).dt.days

if "ad_status" in enr.columns:
    ad_status_norm = enr["ad_status"].astype(str).str.lower()
    is_deleted = (ad_status_norm == "deleted")
else:
    is_deleted = pd.Series(False, index=enr.index)
enr["is_dead"] = (
    (~enr["has_any_event_train"].astype(bool) & (enr["contacts_total"] == 0))
    | is_deleted
).astype("int8")

def _tier(row):
    if row["is_dead"]:
        return "D"
    if row["has_any_event_train"]:
        if pd.notna(row["recency_evt_days"]) and row["recency_evt_days"] <= 30:
            return "A"
        return "B"
    return "C"

enr["tier"] = enr.apply(_tier, axis=1)
print(enr["tier"].value_counts(dropna=False))
print(f"\nA share: {(enr['tier']=='A').mean()*100:.1f}%")
print(f"D share: {(enr['tier']=='D').mean()*100:.1f}%")

In [ ]:
out = f"{CACHE_DIR}/dim_listing_enriched.parquet"
enr.to_parquet(out, index=False)
print(f"Wrote {out} | {enr.shape}")